In [17]:
import pandas as pd
path = "https://raw.githubusercontent.com/ulka29/paper/7aad80ab3fe4ad5ffeabffb7c30bcb70edb218bd/01_data/processed/df_cleaned.csv"# import Check dataset
df= pd.read_csv(path)
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258


In [18]:
# Select relevant features for SA + ML
df_ml = df[
    [
        'lemmatized_tweet',
        'negative',
        'positive',
        'neutral',
        'compound',
        'cyberbullying_type'
    ]
].copy()

# Check shape
print("\n:-------------Shape::-------------", df_ml.shape)

# Check column names
print("\n:-------------Columns::-------------")
print(df_ml.columns.tolist())

# Check missing values
print("\n----------------Missing values:-------------")
print(df_ml.isnull().sum())

# Display first 5 rows
df_ml.head()


:-------------Shape::------------- (42645, 6)

:-------------Columns::-------------
['lemmatized_tweet', 'negative', 'positive', 'neutral', 'compound', 'cyberbullying_type']

----------------Missing values:-------------
lemmatized_tweet      30
negative               0
positive               0
neutral                0
compound               0
cyberbullying_type     0
dtype: int64


,lemmatized_tweet,negative,positive,neutral,compound,cyberbullying_type
0,word katandandre food crapilicious mkr,0.000,0.000,1.000,0.0000,not_cyberbullying
1,aussietv white mkr theblock imacelebrityau tod...,0.000,0.000,1.000,0.0000,not_cyberbullying
2,classy whore red velvet cupcake,0.320,0.199,0.481,-0.4137,not_cyberbullying
3,meh p thanks head concerned another angry dude...,0.243,0.150,0.607,-0.4854,not_cyberbullying
4,isi account pretending kurdish account like is...,0.129,0.180,0.691,0.0258,not_cyberbullying


In [19]:
PROFANITY_WORDS = {
    'fuck', 'fucking', 'fucked', 'fuckin',
    'shit', 'shitty',
    'bitch', 'bitches',
    'asshole', 'assholes',
    'bastard',
    'damn',
    'dick',
    'piss',
    'crap',
    'bullshit',
    'motherfucker',
    'idiot',
    'stupid',
    'moron',
    'retard',
    'retarded'
}

In [20]:
def extract_linguistic_features(text):

    text = str(text)

    # Basic tokens
    words = re.findall(r'\b\w+\b', text.lower())

    word_count = len(words)
    char_count = len(text)
    unique_words = len(set(words))

    # Profanity
    profanity_words = [
        word for word in words
        if word in PROFANITY_WORDS
    ]

    profanity_count = len(profanity_words)
    unique_profanity_count = len(set(profanity_words))

    profanity_ratio = (
        profanity_count / word_count
        if word_count > 0 else 0
    )

    profanity_presence = int(profanity_count > 0)

    # Second-person language
    second_person_words = {
        'you',
        'your',
        'yours',
        'yourself',
        'yourselves',
        'youre',
        'youve',
        'youll',
        'youd'
    }

    second_person_count = sum(
        word in second_person_words
        for word in words
    )

    # Punctuation
    exclamation_count = text.count('!')
    question_count = text.count('?')
    comma_count = text.count(',')
    period_count = text.count('.')

    repeated_exclamation_count = len(
        re.findall(r'!{2,}', text)
    )

    repeated_question_count = len(
        re.findall(r'\?{2,}', text)
    )

    repeated_punctuation_count = len(
        re.findall(r'[!?]{2,}', text)
    )

    # Mentions / Hashtags
    mention_count = len(
        re.findall(r'@\w+', text)
    )

    hashtag_count = len(
        re.findall(r'#\w+', text)
    )

    # Return features
    return pd.Series({
        'word_count': word_count,
        'char_count': char_count,
        'unique_word_count': unique_words,

        'profanity_count': profanity_count,
        'unique_profanity_count': unique_profanity_count,
        'profanity_ratio': profanity_ratio,
        'profanity_presence': profanity_presence,

        'second_person_count': second_person_count,

        'exclamation_count': exclamation_count,
        'question_count': question_count,
        'comma_count': comma_count,
        'period_count': period_count,

        'repeated_exclamation_count': repeated_exclamation_count,
        'repeated_question_count': repeated_question_count,
        'repeated_punctuation_count': repeated_punctuation_count,

        'mention_count': mention_count,
        'hashtag_count': hashtag_count
    })


In [21]:
import re

In [22]:
linguistic_features = df['tweet_text'].apply(
    extract_linguistic_features
)

df = pd.concat(
    [df, linguistic_features],
    axis=1
)

In [23]:
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound,word_count,...,second_person_count,exclamation_count,question_count,comma_count,period_count,repeated_exclamation_count,repeated_question_count,repeated_punctuation_count,mention_count,hashtag_count
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,9.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,14.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137,9.0,...,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0


In [24]:
print("Class-wise counts:")
for class_name, count in df['cyberbullying_type'].value_counts().items():
    print(f"{class_name}: {count}")

Class-wise counts:
religion: 7964
age: 7957
gender: 7491
ethnicity: 7427
not_cyberbullying: 6445
other_cyberbullying: 5361


In [25]:
print(df.columns.tolist())

['tweet_text', 'cyberbullying_type', 'clean_tweet', 'lemmatized_tweet', 'sentiment_scores', 'negative', 'positive', 'neutral', 'compound', 'word_count', 'char_count', 'unique_word_count', 'profanity_count', 'unique_profanity_count', 'profanity_ratio', 'profanity_presence', 'second_person_count', 'exclamation_count', 'question_count', 'comma_count', 'period_count', 'repeated_exclamation_count', 'repeated_question_count', 'repeated_punctuation_count', 'mention_count', 'hashtag_count']


In [26]:
# ============================================================
# CELL 1: DATA PREPARATION
# HEXA CLASS + WORD TF-IDF + SENTIMENT FEATURES
# DATASET: df_ml
# ============================================================

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# ------------------------------------------------------------
# 1. SELECT TEXT, SENTIMENT FEATURES AND TARGET
# ------------------------------------------------------------

text_col = "lemmatized_tweet"
target_col = "cyberbullying_type"

sentiment_features = [
    "negative",
    "positive",
    "neutral",
    "compound"
]

behavioral_features = [
    "second_person_count",
    "mention_count",
    "hashtag_count",
    "profanity_count",
    "unique_profanity_count",
    "profanity_ratio",
    "profanity_presence"
]

# Keep rows with valid target labels
df_exp = df.dropna(subset=[target_col]).copy()

X_text = df_exp[text_col].fillna("").astype(str)

# Convert numeric features safely
all_numeric_features = sentiment_features + behavioral_features

X_numeric = (
    df_exp[all_numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df_exp[target_col].astype(str)

# ------------------------------------------------------------
# 2. LABEL ENCODING
# ------------------------------------------------------------

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes:")
print(label_encoder.classes_)

label_mapping = dict(enumerate(label_encoder.classes_))

print("\nLabel Mapping:")
for number, category in label_mapping.items():
    print(f"{number} -> {category}")

print("\nDataset shape:", df_exp.shape)
print("Number of classes:", len(label_encoder.classes_))

# ------------------------------------------------------------
# 3. FIXED STRATIFIED TRAIN-TEST SPLIT
# ------------------------------------------------------------

(
    X_text_train,
    X_text_test,
    X_num_train,
    X_num_test,
    y_train,
    y_test
) = train_test_split(
    X_text,
    X_numeric,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain-Test Split")
print("-------------------------")
print("Training samples:", len(y_train))
print("Testing samples :", len(y_test))

# ------------------------------------------------------------
# 4. WORD TF-IDF VECTORIZATION
# Fit ONLY on training text
# ------------------------------------------------------------

tfidf = TfidfVectorizer(
    analyzer="word",
    max_features=10000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf = tfidf.transform(X_text_test)

print("\nWord TF-IDF Features")
print("-------------------------")
print("X_train TF-IDF:", X_train_tfidf.shape)
print("X_test TF-IDF :", X_test_tfidf.shape)



# ------------------------------------------------------------
# 4. SCALE NUMERIC FEATURES INDEPENDENTLY
# Fit scaler ONLY on training data
# ------------------------------------------------------------

scaler = StandardScaler()

# Fit on training data; transform train and test separately
X_num_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_num_train),
    columns=X_num_train.columns,
    index=X_num_train.index
)

X_num_test_scaled = pd.DataFrame(
    scaler.transform(X_num_test),
    columns=X_num_test.columns,
    index=X_num_test.index
)

print("\nNumeric Feature Scaling")
print("-------------------------")
print("Training means (approximately 0):")
print(X_num_train_scaled.mean().round(4))

print("\nTraining standard deviations (approximately 1):")
print(X_num_train_scaled.std(ddof=0).round(4))

Classes:
['age' 'ethnicity' 'gender' 'not_cyberbullying' 'other_cyberbullying'
 'religion']

Label Mapping:
0 -> age
1 -> ethnicity
2 -> gender
3 -> not_cyberbullying
4 -> other_cyberbullying
5 -> religion

Dataset shape: (42645, 26)
Number of classes: 6

Train-Test Split
-------------------------
Training samples: 29851
Testing samples : 12794

Word TF-IDF Features
-------------------------
X_train TF-IDF: (29851, 10000)
X_test TF-IDF : (12794, 10000)

Numeric Feature Scaling
-------------------------
Training means (approximately 0):
negative                 -0.0
positive                 -0.0
neutral                  -0.0
compound                  0.0
second_person_count       0.0
mention_count            -0.0
hashtag_count            -0.0
profanity_count           0.0
unique_profanity_count    0.0
profanity_ratio          -0.0
profanity_presence        0.0
dtype: float64

Training standard deviations (approximately 1):
negative                  1.0
positive                  1.0
neut

In [27]:
# ============================================================
# CELL 2: FEATURE CONFIGURATIONS
# EXPERIMENTS E1-E4
# ============================================================

# ------------------------------------------------------------
# 1. DEFINE EXPERIMENTS
# ------------------------------------------------------------

experiments = {
    "E1_Text_Sentiment": [
        "negative",
        "positive",
        "neutral",
        "compound"
    ],

    "E2_Sentiment_Profanity_Behavior": [
        "negative",
        "positive",
        "neutral",
        "compound",
        "second_person_count",
        "profanity_count",
        "unique_profanity_count",
        "profanity_ratio",
        "profanity_presence"
    ],

    "E3_Sentiment_Mention_Hashtag": [
        "negative",
        "positive",
        "neutral",
        "compound",
        "mention_count",
        "hashtag_count"
    ],

    "E4_Sentiment_All_Behavior": [
        "negative",
        "positive",
        "neutral",
        "compound",
        "second_person_count",
        "mention_count",
        "hashtag_count",
        "profanity_count",
        "unique_profanity_count",
        "profanity_ratio",
        "profanity_presence"
    ]
}

# ------------------------------------------------------------
# 2. CREATE COMBINED FEATURE MATRICES
# Same split and same TF-IDF representation for every experiment
# ------------------------------------------------------------

X_train_experiments = {}
X_test_experiments = {}

for exp_name, feature_cols in experiments.items():

    # Select the experiment's numeric features
    # Use scaled numeric features for every experiment
    X_num_train_exp = X_num_train_scaled[feature_cols]
    X_num_test_exp = X_num_test_scaled[feature_cols]

    # X_num_train_exp = X_num_train[feature_cols]
    # X_num_test_exp = X_num_test[feature_cols]

    # Convert numeric features to sparse matrices
    X_num_train_sparse = csr_matrix(
        X_num_train_exp.to_numpy(dtype=float)
    )

    X_num_test_sparse = csr_matrix(
        X_num_test_exp.to_numpy(dtype=float)
    )

    # Combine Word TF-IDF + selected numeric features
    X_train_experiments[exp_name] = hstack(
        [X_train_tfidf, X_num_train_sparse],
        format="csr"
    )

    X_test_experiments[exp_name] = hstack(
        [X_test_tfidf, X_num_test_sparse],
        format="csr"
    )

    print("\n" + "=" * 60)
    print(exp_name)
    print("=" * 60)
    print("Additional numeric features:", len(feature_cols) - 4)
    print("Total numeric features:", len(feature_cols))
    print("Training shape:", X_train_experiments[exp_name].shape)
    print("Testing shape :", X_test_experiments[exp_name].shape)


E1_Text_Sentiment
Additional numeric features: 0
Total numeric features: 4
Training shape: (29851, 10004)
Testing shape : (12794, 10004)

E2_Sentiment_Profanity_Behavior
Additional numeric features: 5
Total numeric features: 9
Training shape: (29851, 10009)
Testing shape : (12794, 10009)

E3_Sentiment_Mention_Hashtag
Additional numeric features: 2
Total numeric features: 6
Training shape: (29851, 10006)
Testing shape : (12794, 10006)

E4_Sentiment_All_Behavior
Additional numeric features: 7
Total numeric features: 11
Training shape: (29851, 10011)
Testing shape : (12794, 10011)


In [28]:
# ============================================================
# CELL 3: HEXA CLASS + WORD TF-IDF + SENTIMENT / BEHAVIORAL
# NO BALANCING + LOGISTIC REGRESSION
# FIXED HYPERPARAMETERS: C=1, max_iter=1000
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. FIXED MODEL CONFIGURATION
# ------------------------------------------------------------

lr_template = LogisticRegression(
    C=1.0,
    class_weight=None,
    max_iter=1000,
    random_state=42
)

# ------------------------------------------------------------
# 2. TRAIN AND EVALUATE ALL EXPERIMENTS
# ------------------------------------------------------------

results = []
fitted_models = {}
predictions = {}
reports = {}
confusion_matrices = {}

for exp_name in experiments:

    print("\n" + "=" * 65)
    print(exp_name)
    print("=" * 65)

    # Create a fresh model for each experiment
    model = LogisticRegression(
        C=1.0,
        class_weight=None,
        max_iter=1000,
        random_state=42
    )

    # Train
    print("Training Logistic Regression...")
    model.fit(
        X_train_experiments[exp_name],
        y_train
    )

    # Predict
    y_pred = model.predict(
        X_test_experiments[exp_name]
    )

    # Save fitted model and predictions
    fitted_models[exp_name] = model
    predictions[exp_name] = y_pred

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    results.append({
        "Experiment": exp_name,
        "Numeric Features": len(experiments[exp_name]),
        "Accuracy": accuracy,
        "Macro-F1": macro_f1,
        "Weighted-F1": weighted_f1
    })

    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Macro-F1    : {macro_f1:.4f}")
    print(f"Weighted-F1 : {weighted_f1:.4f}")

    # Classification report
    report = classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0
    )

    reports[exp_name] = report

    print("\nClassification Report")
    print("-------------------------")

    print(classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        zero_division=0
    ))

    # Confusion matrix
    confusion_matrices[exp_name] = confusion_matrix(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_))
    )

# ------------------------------------------------------------
# 3. OVERALL COMPARISON TABLE
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Macro-F1",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 75)
print("EXPERIMENT COMPARISON: LOGISTIC REGRESSION, NO BALANCING")
print("=" * 75)

display(results_df.round(4))


E1_Text_Sentiment
Training Logistic Regression...
Accuracy    : 0.8493
Macro-F1    : 0.8304
Weighted-F1 : 0.8502

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.95      0.97      0.96      2387
          ethnicity       0.97      0.98      0.98      2228
             gender       0.92      0.85      0.88      2248
  not_cyberbullying       0.60      0.63      0.62      1934
other_cyberbullying       0.59      0.59      0.59      1608
           religion       0.95      0.95      0.95      2389

           accuracy                           0.85     12794
          macro avg       0.83      0.83      0.83     12794
       weighted avg       0.85      0.85      0.85     12794


E2_Sentiment_Profanity_Behavior
Training Logistic Regression...
Accuracy    : 0.8503
Macro-F1    : 0.8313
Weighted-F1 : 0.8511

Classification Report
-------------------------
                     precision    recall  f1-sc

,Experiment,Numeric Features,Accuracy,Macro-F1,Weighted-F1
0,E4_Sentiment_All_Behavior,11,0.8503,0.8315,0.8513
1,E2_Sentiment_Profanity_Behavior,9,0.8503,0.8313,0.8511
2,E3_Sentiment_Mention_Hashtag,6,0.8494,0.8304,0.8503
3,E1_Text_Sentiment,4,0.8493,0.8304,0.8502


In [ ]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [ ]:

# ============================================================
# EXPERIMENT 2 — MODEL COMPARISON + HYPERPARAMETER TUNING
# Objective: Compare 5 ML/DL models using 3-fold CV Macro-F1
# Input: Existing TF-IDF + numeric feature matrices
# ============================================================

import time
import warnings
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from IPython.display import display

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    ParameterGrid
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# 1. Validate existing notebook variables
# ------------------------------------------------------------

required_variables = [
    "X_train", "X_test", "y_train", "y_test", "label_encoder"
]

missing_variables = [
    variable for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise NameError(
        f"Missing variables: {missing_variables}. "
        "Run your baseline preprocessing and train/test split cells first."
    )

if X_train.shape[0] != len(y_train):
    raise ValueError("X_train and y_train have different sample counts.")

if X_test.shape[0] != len(y_test):
    raise ValueError("X_test and y_test have different sample counts.")

if X_train.shape[1] != X_test.shape[1]:
    raise ValueError("X_train and X_test have different feature counts.")

# XGBoost multiclass classification expects encoded labels 0, 1, ..., K-1.
expected_labels = np.arange(len(label_encoder.classes_))
actual_labels = np.sort(np.unique(np.asarray(y_train)))

if not np.array_equal(actual_labels, expected_labels):
    raise ValueError(
        "Labels must be encoded as consecutive integers starting at 0. "
        "Check that label_encoder was fitted on the full target column."
    )

print("Training shape:", X_train.shape)
print("Testing shape :", X_test.shape)
print("Classes       :", list(label_encoder.classes_))

# ------------------------------------------------------------
# 2. Define models and hyperparameter grids
# ------------------------------------------------------------

model_configs = {
    "1. XGBoost": {
        "model": XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [3, 6],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8],
            "colsample_bytree": [0.8]
        }
    },

    "2. SGD Classifier": {
        "model": SGDClassifier(
            class_weight="balanced",
            max_iter=10000,
            tol=1e-3,
            random_state=42
        ),
        "params": {
            "loss": ["hinge", "log_loss"],
            "alpha": [1e-4, 1e-3, 1e-2],
            "penalty": ["l2"]
        }
    },

    "3. Random Forest": {
        "model": RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [20, 40],
            "min_samples_leaf": [1, 2]
        }
    },

    "4. Extra Trees": {
        "model": ExtraTreesClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [20, 40],
            "min_samples_leaf": [1, 2]
        }
    },

    "5. MLP Neural Network": {
        "model": Pipeline([
            ("scaler", MaxAbsScaler()),
            ("mlp", MLPClassifier(
                solver="adam",
                activation="relu",
                batch_size=128,
                max_iter=100,
                early_stopping=True,
                n_iter_no_change=8,
                random_state=42
            ))
        ]),
        "params": {
            "mlp__hidden_layer_sizes": [(64,), (128,)],
            "mlp__alpha": [1e-4, 1e-3]
        }
    }
}

# ------------------------------------------------------------
# 3. Configure stratified cross-validation
# ------------------------------------------------------------

cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

scoring_metric = "f1_macro"

# ------------------------------------------------------------
# 4. Prepare result containers
# ------------------------------------------------------------

tuned_results = []
tuned_models = {}
tuned_predictions = {}
tuned_reports = {}
tuned_confusion_matrices = {}
grid_search_objects = {}

# Explicit class order for reports and confusion matrices
class_labels = np.arange(len(label_encoder.classes_))
class_names = [str(name) for name in label_encoder.classes_]

# ------------------------------------------------------------
# 5. Show estimated computational workload
# ------------------------------------------------------------

total_cv_fits = 0

for config in model_configs.values():
    combinations = len(ParameterGrid(config["params"]))
    total_cv_fits += combinations * cv_strategy.get_n_splits()

total_final_refits = len(model_configs)

print(f"\nModels to tune          : {len(model_configs)}")
print(f"Total CV fits           : {total_cv_fits}")
print(f"Final best-model refits : {total_final_refits}")
print(f"Estimated total fits    : {total_cv_fits + total_final_refits}")
print(f"CV scoring metric       : {scoring_metric}")
print("\nNote: actual runtime depends on your machine and dataset size.")

# ------------------------------------------------------------
# 6. Run grid search for each model
# ------------------------------------------------------------

for model_name, config in tqdm(
    model_configs.items(),
    total=len(model_configs),
    desc="Overall Grid Search",
    unit="model"
):
    print("\n" + "=" * 75)
    print(f"MODEL: {model_name}")
    print("=" * 75)

    model = config["model"]
    param_grid = config["params"]

    n_combinations = len(ParameterGrid(param_grid))
    n_splits = cv_strategy.get_n_splits(X_train, y_train)
    expected_cv_fits = n_combinations * n_splits

    print("Parameter combinations:", n_combinations)
    print("CV folds               :", n_splits)
    print("CV fits                :", expected_cv_fits)
    print("Final refit            : 1")

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scoring_metric,
        cv=cv_strategy,
        refit=True,
        n_jobs=1,
        verbose=2,
        error_score="raise",
        return_train_score=False
    )

    start_time = time.time()

    # Fit only on training data; GridSearchCV handles CV internally.
    search.fit(X_train, y_train)

    elapsed_time = time.time() - start_time

    best_model = search.best_estimator_

    # Final evaluation on the held-out test set
    y_pred = best_model.predict(X_test)

    test_accuracy = accuracy_score(y_test, y_pred)
    test_macro_f1 = f1_score(
        y_test, y_pred, average="macro", zero_division=0
    )
    test_weighted_f1 = f1_score(
        y_test, y_pred, average="weighted", zero_division=0
    )

    # Full class-wise report
    report_dict = classification_report(
        y_test,
        y_pred,
        labels=class_labels,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_test,
        y_pred,
        labels=class_labels,
        target_names=class_names,
        zero_division=0
    )

    # Confusion matrix with consistent class order
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=class_labels
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual: {name}" for name in class_names],
        columns=[f"Predicted: {name}" for name in class_names]
    )

    # Store all outputs
    tuned_models[model_name] = best_model
    tuned_predictions[model_name] = y_pred
    tuned_reports[model_name] = report_dict
    tuned_confusion_matrices[model_name] = cm_df
    grid_search_objects[model_name] = search

    tuned_results.append({
        "Model": model_name,
        "Best CV Macro-F1": search.best_score_,
        "Test Accuracy": test_accuracy,
        "Test Macro-F1": test_macro_f1,
        "Test Weighted-F1": test_weighted_f1,
        "Best Parameters": search.best_params_,
        "CV Fits": expected_cv_fits,
        "Runtime (minutes)": elapsed_time / 60
    })

    print("\nBest parameters:")
    print(search.best_params_)

    print(f"\nBest CV Macro-F1 : {search.best_score_:.4f}")
    print(f"Test Accuracy    : {test_accuracy:.4f}")
    print(f"Test Macro-F1    : {test_macro_f1:.4f}")
    print(f"Test Weighted-F1 : {test_weighted_f1:.4f}")
    print(f"Runtime          : {elapsed_time / 60:.2f} minutes")

    print("\nClassification Report:")
    print(report_text)

    print("\nConfusion Matrix:")
    display(cm_df)

# ------------------------------------------------------------
# 7. Compare models
# ------------------------------------------------------------

tuned_results_df = pd.DataFrame(tuned_results)

# Rank using CV Macro-F1, not held-out test performance.
tuned_results_df = tuned_results_df.sort_values(
    by="Best CV Macro-F1",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 90)
print("FINAL MODEL COMPARISON")
print("=" * 90)

display(
    tuned_results_df[
        [
            "Model",
            "Best CV Macro-F1",
            "Test Accuracy",
            "Test Macro-F1",
            "Test Weighted-F1",
            "Runtime (minutes)"
        ]
    ].round(4)
)

print("\nBest hyperparameters for each model:")
for _, row in tuned_results_df.iterrows():
    print(f"\n{row['Model']}")
    print(row["Best Parameters"])

# ------------------------------------------------------------
# 8. Inspect Not-Cyberbullying performance
# ------------------------------------------------------------

not_cb_class = "not_cyberbullying"

if not_cb_class in class_names:
    print("\n" + "=" * 90)
    print("NOT-CYBERBULLYING CLASS PERFORMANCE")
    print("=" * 90)

    not_cb_results = []

    for model_name, report in tuned_reports.items():
        class_metrics = report.get(not_cb_class, {})

        not_cb_results.append({
            "Model": model_name,
            "Not-CB Precision": class_metrics.get("precision", np.nan),
            "Not-CB Recall": class_metrics.get("recall", np.nan),
            "Not-CB F1": class_metrics.get("f1-score", np.nan),
            "Not-CB Support": class_metrics.get("support", np.nan)
        })

    not_cb_results_df = pd.DataFrame(not_cb_results)

    not_cb_results_df = not_cb_results_df.sort_values(
        by="Not-CB F1",
        ascending=False
    ).reset_index(drop=True)

    display(not_cb_results_df.round(4))

else:
    print(
        f"\n'{not_cb_class}' was not found in label_encoder.classes_. "
        "Check the original target labels."
    )

print("\nExperiment 2 complete.")


Training shape: (29851, 10011)
Testing shape : (12794, 10011)
Classes       : ['age', 'ethnicity', 'gender', 'not_cyberbullying', 'other_cyberbullying', 'religion']

Models to tune          : 5
Total CV fits           : 102
Final best-model refits : 5
Estimated total fits    : 107
CV scoring metric       : f1_macro

Note: actual runtime depends on your machine and dataset size.


Overall Grid Search:   0%|          | 0/5 [00:00<?, ?model/s]


MODEL: 1. XGBoost
Parameter combinations: 8
CV folds               : 3
CV fits                : 24
Final refit            : 1
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=100, subsample=0.8; total time=  27.8s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=100, subsample=0.8; total time=  27.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=100, subsample=0.8; total time=  26.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.8; total time=  49.8s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.8; total time=  50.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.8; total time=  50.0s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=100, subsample=0.8; total time= 1.6min
[

,Predicted: age,Predicted: ethnicity,Predicted: gender,Predicted: not_cyberbullying,Predicted: other_cyberbullying,Predicted: religion
Actual: age,2313,2,1,44,27,0
Actual: ethnicity,1,2193,4,10,19,1
Actual: gender,2,2,1903,237,102,2
Actual: not_cyberbullying,17,7,77,1362,409,62
Actual: other_cyberbullying,8,6,67,606,917,4
Actual: religion,2,1,3,82,29,2272



MODEL: 2. SGD Classifier
Parameter combinations: 6
CV folds               : 3
CV fits                : 18
Final refit            : 1
Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END ...............alpha=0.0001, loss=hinge, penalty=l2; total time=   0.5s
[CV] END ...............alpha=0.0001, loss=hinge, penalty=l2; total time=   0.6s
[CV] END ...............alpha=0.0001, loss=hinge, penalty=l2; total time=   0.5s
[CV] END ............alpha=0.0001, loss=log_loss, penalty=l2; total time=   0.5s
[CV] END ............alpha=0.0001, loss=log_loss, penalty=l2; total time=   0.6s
[CV] END ............alpha=0.0001, loss=log_loss, penalty=l2; total time=   0.6s
[CV] END ................alpha=0.001, loss=hinge, penalty=l2; total time=   0.3s
[CV] END ................alpha=0.001, loss=hinge, penalty=l2; total time=   0.3s
[CV] END ................alpha=0.001, loss=hinge, penalty=l2; total time=   0.3s
[CV] END .............alpha=0.001, loss=log_loss, penalty=l2; total time=   0

...


FINAL MODEL COMPARISON


,Model,Best CV Macro-F1,Test Accuracy,Test Macro-F1,Test Weighted-F1,Runtime (minutes)
0,1. XGBoost,0.8296,0.8567,0.8397,0.8597,35.1257
1,2. SGD Classifier,0.8202,0.8477,0.8271,0.8464,0.1312
2,3. Random Forest,0.8196,0.8426,0.8267,0.8471,2.9685
3,5. MLP Neural Network,0.8097,0.8427,0.8216,0.8420,12.3594
4,4. Extra Trees,0.7904,0.8237,0.7958,0.8212,1.7436



Best hyperparameters for each model:

1. XGBoost
{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}

2. SGD Classifier
{'alpha': 0.0001, 'loss': 'hinge', 'penalty': 'l2'}

3. Random Forest
{'max_depth': 40, 'min_samples_leaf': 1, 'n_estimators': 200}

5. MLP Neural Network
{'mlp__alpha': 0.001, 'mlp__hidden_layer_sizes': (128,)}

4. Extra Trees
{'max_depth': 40, 'min_samples_leaf': 1, 'n_estimators': 200}

NOT-CYBERBULLYING CLASS PERFORMANCE


,Model,Not-CB Precision,Not-CB Recall,Not-CB F1,Not-CB Support
0,1. XGBoost,0.5818,0.7042,0.6372,1934.0
1,4. Extra Trees,0.5182,0.7746,0.6209,1934.0
2,3. Random Forest,0.5795,0.6334,0.6052,1934.0
3,5. MLP Neural Network,0.5947,0.6138,0.6041,1934.0
4,2. SGD Classifier,0.6151,0.5760,0.5949,1934.0



Experiment 2 complete.
